# Spend Analytics: ML development on Cloudera AI

The nightly pipeline runs three CAI jobs: `spend-feature-engineering` → `spend-model-training` →
`spend-model-evaluation`. This notebook runs **the same three functions** (`spend/pipeline.py`) step by
step on today's gold data, and opens up each model in between.

| Step | Job it mirrors | What it does here |
|---|---|---|
| 1 | `spend-feature-engineering` | pins the Iceberg snapshots, reads the features and cohort series |
| 2 | `spend-model-training` | personas (KMeans), categoriser (MiniLM + n-grams), forecast (Chronos-2), signals |
| 3 | `spend-model-evaluation` | the KPI gate, **dry run** |

**Nothing is published.** Stage hand-off goes to `artifacts/demo/`, apart from the nightly
`artifacts/<date>/`. Evaluation runs with `write=False`: no gold table, MLflow run or registry version is
written. Session: Python 3.11, 4 vCPU / 16 GB. Avoid 02:30-03:20 IST, while the nightly DAG runs.

In [ ]:
import json
import logging
import os
import sys
import time
from datetime import date
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HOME", str(ROOT / ".hf_cache"))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from spend import gate as kpi_gate
from spend.categorise import SeedIndex, assign, load_embedder, normalise
from spend.config import policy
from spend.forecast import load_forecaster
from spend.pipeline import artifacts_dir, evaluate_models, feature_engineering, train_models
from spend.storage import get_storage

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
pd.set_option("display.max_colwidth", 80)

DEMO_ROOT = ROOT / "artifacts" / "demo"
DEMO_CIF = "C00016807"        # planted demo customer: Bengaluru affluent foodie with a Goa trip
POLICY = policy()
storage = get_storage()       # impala on CAI; SPEND_STORAGE_BACKEND=parquet on a laptop
print(f"backend: {storage.name} | hand-off folder: {DEMO_ROOT}")

## Step 1. Feature engineering

Reads the latest snapshot of `gold.spend_features` (one row per customer, 13-week and 4-week behaviour),
the 4-weeks-earlier snapshot for the relocation signal, the cohort × category weekly series and the
categoriser holdouts. Every Iceberg snapshot ID is pinned in `manifest.json`, so the run is reproducible
with `FOR SYSTEM_VERSION AS OF <id>`.

In [ ]:
t = time.time()
manifest = feature_engineering(storage, triggered_by="notebook-demo", root=DEMO_ROOT)
RUN_DATE = date.fromisoformat(manifest["run_date"])
D = artifacts_dir(RUN_DATE, DEMO_ROOT)
print(f"prepared in {time.time() - t:.1f}s -> {D}")

keys = ["run_id", "run_date", "prior_snapshot_date", "features_snapshot_id", "cohort_snapshot_id",
        "map_snapshot_id", "n_customers", "n_series", "cohort_weeks", "map_keys", "uncategorised_share",
        "prev_run_date", "champion_run_id", "feature_hash"]
pd.DataFrame({"value": [str(manifest[k]) for k in keys]}, index=keys)

In [ ]:
feats = pd.read_parquet(D / "features.parquet")
print(f"{len(feats):,} customers x {feats.shape[1]} columns")
cols = ["segment", "home_city", "city_tier", "spend_13w", "txn_13w", "digital_share", "home_share",
        "cat_travel_share", "cat_food_delivery_share", "cat_dining_share", "cat_travel_4w", "away_share_4w"]
feats.set_index("cif").loc[[DEMO_CIF], cols].T

## Step 2. Training

Both pretrained models are loaded once (cached in `HF_HOME`; CPU is enough). `train_models` is the whole
`spend-model-training` job: about 45 s on 4 vCPU. The sections below then open up each output.

In [ ]:
t = time.time()
embedder, forecaster = load_embedder(), load_forecaster()
print(f"{embedder.model_id} | {forecaster.model_id} on {getattr(forecaster, 'device', 'cpu')} "
      f"loaded in {time.time() - t:.1f}s")

t = time.time()
metrics = train_models(RUN_DATE, root=DEMO_ROOT, embedder=embedder, forecaster=forecaster)
print(f"trained in {time.time() - t:.1f}s; stage seconds: {metrics['stage_durations']}")

### 2a. Personas: KMeans on 27 standardised features

`k` is searched over 5-9 and chosen by silhouette score. Persona IDs are matched to the previous run
(Hungarian algorithm), so a persona keeps its ID and name from week to week.

In [ ]:
sil = pd.Series(metrics["silhouette_by_k"]).rename_axis("k").reset_index(name="silhouette")
fig = px.bar(sil, x="k", y="silhouette", title=f"Silhouette by k (chosen: k = {metrics['k']})")
fig.add_hline(y=POLICY["gate"]["min_silhouette"], line_dash="dash", annotation_text="gate floor")
fig.show()

print(f"stability vs previous run: {metrics['persona_stability']} | "
      f"ARI vs the planted archetypes: {metrics['ari_hidden']}")
profile = pd.read_parquet(D / "persona_profile.parquet")
profile[["persona_id", "persona_name", "n_customers", "customer_share", "avg_spend_13w", "top_category",
         "top_channel", "digital_share", "home_share"]].sort_values("n_customers", ascending=False).round(3)

In [ ]:
cp = pd.read_parquet(D / "customer_persona.parquet")
display(cp[cp["cif"] == DEMO_CIF][["cif", "home_city", "segment", "persona_name", "prev_persona_id",
                                   "top_category", "spend_13w", "distance"]])

flow_path = D / "persona_flow.parquet"
if flow_path.exists():
    flow = pd.read_parquet(flow_path)
    names = sorted(set(flow["from_persona_name"]) | set(flow["to_persona_name"]))
    idx = {n: i for i, n in enumerate(names)}
    fig = go.Figure(go.Sankey(
        node=dict(label=[f"{n} (prev)" for n in names] + names, pad=12),
        link=dict(source=flow["from_persona_name"].map(idx), target=flow["to_persona_name"].map(idx) + len(names),
                  value=flow["n_customers"])))
    fig.update_layout(title=f"Persona flow since {flow['prev_run_date'].iloc[0]}", height=550)
    fig.show()

### 2b. Categoriser: MiniLM embeddings + character n-grams

UPI payments carry only a payee name, with no MCC. Accuracy is measured two ways: on 20% of seed merchants
held out of the index, and on card merchants with their MCC hidden.

In [ ]:
seed_h, mcc_h = pd.read_parquet(D / "seed_holdout.parquet"), pd.read_parquet(D / "mcc_holdout.parquet")
display(pd.DataFrame({"accuracy": [metrics["category_accuracy_seed"], metrics["category_accuracy_mcc"]],
                      "n": [metrics["n_seed_holdout"], metrics["n_mcc_holdout"]],
                      "gate floor": POLICY["gate"]["min_category_accuracy"]},
                     index=["seed holdout", "hidden MCC"]).round(3))

by_cat = (mcc_h.assign(correct=mcc_h["category"] == mcc_h["truth"]).groupby("truth")["correct"]
          .agg(["mean", "size"]).sort_values("mean").reset_index())
px.bar(by_cat, x="mean", y="truth", orientation="h", hover_data=["size"], range_x=[0, 1],
       title="Hidden-MCC accuracy by true category").show()

print("sample misses:")
mcc_h[mcc_h["category"] != mcc_h["truth"]].head(8)[["merchant_key", "truth", "category", "method",
                                                     "similarity", "nearest_seed"]]

In [ ]:
seeds, taxonomy = pd.read_parquet(D / "merchant_seed.parquet"), pd.read_parquet(D / "category_taxonomy.parquet")
index = SeedIndex.build(seeds, taxonomy, embedder)

texts = ["SWIGGY*BLR 5521", "ola cabs pvt", "SHARMA MEDICOS", "RAMESH KIRANA STORE", "PAYTM*BESCOM BILL",
         "TANISHQ JEWELLERS"]   # try your own
live = assign([normalise(x) for x in texts], index, embedder, POLICY["categorise"])
live.insert(0, "payee text", texts)
live.round(3)

### 2c. Forecast: Chronos-2 vs seasonal naive

224 cohort (segment × city tier) × category weekly series, forecast 12 weeks ahead with calendar
covariates (festivals, salary weeks, school fees). The backtest hides the next 12 weeks at 4 rolling
cut-offs and scores WAPE against a same-week-last-year × trend baseline.

In [ ]:
bt = pd.read_parquet(D / "spend_backtest.parquet")
err = bt.assign(model=(bt["actual"] - bt["p50"]).abs(), naive=(bt["actual"] - bt["naive"]).abs())
w = err.groupby("category")[["model", "naive", "actual"]].sum()
w = pd.DataFrame({"Chronos-2": w["model"] / w["actual"], "seasonal naive": w["naive"] / w["actual"]})
px.bar(w.reset_index().melt(id_vars="category", var_name="model", value_name="WAPE"), x="category", y="WAPE",
       color="model", barmode="group", title=f"Backtest WAPE by category: ratio {metrics['wape_ratio']:.3f} "
       f"(gate <= {POLICY['gate']['max_wape_vs_naive']}), P10-P90 coverage {metrics['coverage']:.3f}").show()

In [ ]:
demo = feats.set_index("cif").loc[DEMO_CIF]
SERIES = f"{demo['cohort']}|TRAVEL"          # the demo customer's cohort; try another category
hist = pd.read_parquet(D / "cohort_weekly.parquet")
hist = hist[(hist["cohort"] + "|" + hist["category"]) == SERIES].sort_values("week_end")
fc_ = pd.read_parquet(D / "spend_forecast.parquet").query("series_id == @SERIES").sort_values("week_end")

fig = go.Figure()
fig.add_scatter(x=hist["week_end"], y=hist["amount_inr"], name="actual", line=dict(color="black"))
fig.add_scatter(x=fc_["week_end"], y=fc_["p90"], line=dict(width=0), showlegend=False)
fig.add_scatter(x=fc_["week_end"], y=fc_["p10"], fill="tonexty", line=dict(width=0), name="P10-P90",
                fillcolor="rgba(31,119,180,0.25)")
fig.add_scatter(x=fc_["week_end"], y=fc_["p50"], name="Chronos-2 P50", line=dict(color="#1f77b4"))
fig.add_scatter(x=fc_["week_end"], y=fc_["naive"], name="seasonal naive", line=dict(dash="dot", color="gray"))
fig.update_layout(title=f"{SERIES}: 12-week forecast (INR / week)", height=450)
fig.show()

### 2d. Signals: policy rules, not a model

Rules in `config/policy.yaml` (owned by marketing analytics) applied to today's features. Changing a
threshold re-lists customers on the next run without retraining anything.

In [ ]:
sig = pd.read_parquet(D / "spend_signals.parquet")
display(sig.groupby(["signal_id", "action"]).size().rename("customers").reset_index())
sig[sig["cif"] == DEMO_CIF][["cif", "signal_id", "metric", "metric_value", "threshold", "detail", "persona_name"]]

## Step 3. Evaluation: the KPI gate (dry run)

The gate checks each KPI against `config/policy.yaml`, plus non-regression against the champion (the
published run this candidate would replace). The nightly job publishes only on PASS. A FAIL keeps
yesterday's personas, forecast and signals live, and the DAG goes red. `write=False`: nothing is written.

In [ ]:
result = evaluate_models(storage, RUN_DATE, write=False, tracker=None, registrar=None, root=DEMO_ROOT)
print("GATE", "PASSED: would publish" if result["gate_passed"] else "FAILED: would keep yesterday's run")
pd.DataFrame([(line[:4], line[5:]) for line in result["gate_detail"]], columns=["result", "check"])

**What a rejection looks like.** The same metrics against a stricter gate, built in memory only;
`config/policy.yaml` is not touched.

In [ ]:
strict = {**POLICY["gate"], "min_category_accuracy": 0.95, "max_wape_vs_naive": 0.70}
passed, lines = kpi_gate.evaluate({**metrics, "feature_hash": manifest["feature_hash"]}, strict,
                                  code_hash=manifest["feature_hash"], champion=manifest.get("champion_kpis"))
print("GATE", "PASSED" if passed else "REJECTED the candidate")
pd.DataFrame([(line[:4], line[5:]) for line in lines], columns=["result", "check"])

## In production: every night, the same code

What the nightly DAG published, read back from `gold.spend_model_run`. Then the CAI tabs to show:
**Experiments › spend-analytics** (one MLflow run per night with these KPIs and the persona model),
**Registered Models › spend-persona** (a champion version per passed run) and **Model Deployments ›
spend-scorer** (serves the latest published run).

In [ ]:
runs = storage.read("spend_model_run")
runs = runs.sort_values(["run_date", "run_ts"]).drop_duplicates("run_date", keep="last").tail(15)
kpis = ["category_accuracy_seed", "category_accuracy_mcc", "wape_ratio", "coverage", "silhouette",
        "persona_stability"]
display(runs[["run_date", "run_id", "triggered_by", "gate_passed", "k", *kpis]].round(3))
px.line(runs.melt(id_vars="run_date", value_vars=kpis, var_name="kpi"), x="run_date", y="value", color="kpi",
        markers=True, title="KPIs of the published runs").show()

In [ ]:
try:
    import mlflow
    ml = mlflow.search_runs(experiment_names=["spend-analytics"], order_by=["start_time DESC"], max_results=10)
    cols = [c for c in ml.columns if c in ("run_id", "start_time", "status") or c.startswith("metrics.")][:10]
    display(ml[cols])
except Exception as e:
    print(f"MLflow not available here ({type(e).__name__}): open Experiments > spend-analytics in CAI")

### The deployed model: `spend-scorer`

The same artifacts behind a REST endpoint (needs `SPEND_ENDPOINT_URL`, `_ACCESS_KEY`, `_API_KEY` in the
project environment). Profile the demo customer, then the what-if without his Goa trip.

In [ ]:
from spend.client import call_endpoint, endpoint_configured

if endpoint_configured():
    for req in ({"action": "profile", "cif": DEMO_CIF},
                {"action": "what_if", "cif": DEMO_CIF, "shift": {"TRAVEL": 0.0}}):
        t = time.time()
        resp = call_endpoint(req)
        print(f"# {req['action']}: {time.time() - t:.2f}s")
        print(json.dumps(resp, indent=1, default=str)[:1500])
else:
    print("SPEND_ENDPOINT_URL not set: use the Test tab of spend-scorer instead")